# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This step helps identify how the data is structured in the Croissant schema.

In [ ]:
# List all record sets found in the dataset by their @id, name, and description
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"@id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(unnamed)')}")
    print(f"  Description: {rs.get('description', '')}")
    print(f"  Fields:")
    for field in rs.get('field', []):
        print(f"    - {field['@id']}: {field.get('name', field['@id'])}")
    print()

# For notebook clarity, extract all record set and field @ids for future reference
record_set_ids = [rs['@id'] for rs in record_sets]


## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
dataframes = dict()

for rs_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for: {rs_id}, shape={df.shape}")
    else:
        print(f"No records found for: {rs_id}")

# Example: Print columns from the first non-empty DataFrame, and show a preview
first_df_id = next((rs_id for rs_id in record_set_ids if rs_id in dataframes and not dataframes[rs_id].empty), None)
if first_df_id is not None:
    print(f"Columns for record set {first_df_id}:")
    print(dataframes[first_df_id].columns.tolist())
    display(dataframes[first_df_id].head())
else:
    print("No dataframes with records available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations can include removing outliers, transforming data, or grouping by fields using their `@id`.

In [ ]:
# Choose a numeric field for demonstration (replace with relevant field @id from Data Overview)
if first_df_id is not None:
    df = dataframes[first_df_id]
    # Try to automatically pick the first float or integer column
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]  # Example: pick the first numeric field
        print(f"Numeric field chosen: {numeric_field}\n")

        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely category field (first non-numeric field)
        non_numeric_cols = [col for col in df.columns if col not in numeric_cols]
        if non_numeric_cols:
            group_field = non_numeric_cols[0]
            print(f"\nGrouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No non-numeric field found for grouping.")
    else:
        print("No numeric field found for analysis in this record set.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using record set and field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple histogram and boxplot for numeric column, and a barplot for grouped means
if first_df_id is not None and 'numeric_field' in locals():
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(data=filtered_df, x=numeric_field, kde=True)
    plt.title(f"Distribution of {numeric_field}")

    plt.subplot(1, 2, 2)
    sns.boxplot(data=filtered_df, x=numeric_field)
    plt.title(f"Boxplot of {numeric_field}")
    plt.tight_layout()
    plt.show()

    # If grouped_df exists, visualize category means
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Not enough data to visualize.")

## 6. Conclusion
In this notebook, we loaded, explored, and visualized the FAIR² dataset's record sets and fields as described by the [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json). We referenced all entities by their `@id` as required by the Croissant standard, and demonstrated how to filter, normalize, and group data using the `mlcroissant` library alongside pandas and seaborn for analysis and visualization.

For more advanced analyses, you may use field `@id`s directly from the Data Overview output above to extract and analyze specific features or relationships within the dataset relevant to your domain.
